# A1 reproduction

This notebook reproduces Experiment A1. The canonical experiment measures the number of ordinary Lewis fixed-point updates needed to reach fixed successive-iterate residual thresholds on 50 fixed Gaussian matrices over 15 values of $p$.

The final implementation uses the rescaled-QR evaluation documented in `A1_protocol_record.md` and `A1_EXPERIMENT_RECORD.md`. Fresh outputs go to `reproduced/`; the released files in `data/` and `figures/` are never overwritten.


In [ ]:
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data"
FIGURES = ROOT / "figures"
REPRODUCED = ROOT / "reproduced"
REPRODUCED.mkdir(exist_ok=True)

assert DATA.exists(), "Run this notebook from the public A1 directory."

RUN_FRESH = True

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Fresh rerun enabled:", RUN_FRESH)


## Recorded design

We use 50 fixed $100\times20$ i.i.d. Gaussian matrices, the same matrix bank at every $p$, uniform initialization $w_i^{(0)}=0.2$, residual $R\_t=\max_i|\log(w_i^{(t+1)}/w_i^{(t)})|$, thresholds $10^{-4}$, $10^{-6}$, and $10^{-8}$, and a 10,000-update cap.


In [ ]:
M, N = 100, 20

FINAL_SEEDS = [
    261, 397, 90, 556, 543, 461, 139, 172, 173, 346,
    208, 636, 293, 100, 248, 130, 412, 608, 488, 204,
    748, 314, 671, 592, 493, 576, 389, 440, 552, 596,
    209, 822, 232, 260, 635, 160, 64, 219, 229, 231,
    501, 722, 1389, 442, 413, 512, 559, 563, 882, 697,
]

P_GRID = [
    3, 4, 5.7, 7.3, 10.8, 14.2, 19.6, 27.9,
    32, 38.4, 54.1, 64, 74.8, 98.6, 127.3,
]

THRESHOLDS = (1e-4, 1e-6, 1e-8)
PRIMARY_TOL = 1e-8
MAX_ITER = 10_000

assert len(FINAL_SEEDS) == 50 and len(set(FINAL_SEEDS)) == 50
assert len(P_GRID) == 15 and all(p > 2 for p in P_GRID)


## Released original data

We first verify the released canonical dataset and recompute its summary tables directly from the run-level data. These checks do not rerun the numerical iteration.


In [ ]:
original_bank = np.load(DATA / "A1_matrix_bank.npz")
original_seeds = original_bank["seeds"]
original_matrices = original_bank["matrices"]

original_diag = pd.read_csv(DATA / "A1_matrix_diagnostics.csv")
original_runs = pd.read_csv(DATA / "A1_run_summary.csv")
original_residuals = pd.read_csv(DATA / "A1_residual_trajectories.csv")
released_primary = pd.read_csv(DATA / "A1_primary_summary.csv")
released_tolerance = pd.read_csv(DATA / "A1_tolerance_summary.csv")

assert original_matrices.shape == (50, M, N)
assert np.array_equal(original_seeds, np.array(FINAL_SEEDS, dtype=np.int64))
assert len(original_runs) == 50 * 15
assert original_runs[["matrix_id", "p"]].duplicated().sum() == 0
assert set(original_runs["p"]) == set(P_GRID)
assert original_runs["reached_primary_tol"].all()
assert original_runs["T_1e-8"].notna().all()
assert (original_runs["updates_run"] == original_runs["T_1e-8"]).all()

trajectory_lengths = original_residuals.groupby(["matrix_id", "p"]).size()
expected_lengths = original_runs.set_index(["matrix_id", "p"])["updates_run"]
assert trajectory_lengths.equals(expected_lengths.astype("int64"))

print("PASS: released canonical data contain all 750 cases.")
print("PASS: every canonical run reached 1e-8 before the cap.")
print("Residual rows:", len(original_residuals))


In [ ]:
def summarize_primary(run_df):
    return (
        run_df.groupby("p")["T_1e-8"]
        .agg(
            n="count",
            q20=lambda x: x.quantile(0.20),
            median="median",
            q80=lambda x: x.quantile(0.80),
            min="min",
            max="max",
        )
        .reset_index()
    )


def summarize_tolerances(run_df):
    return (
        run_df.groupby("p")[["T_1e-4", "T_1e-6", "T_1e-8"]]
        .median()
        .reset_index()
    )


original_primary = summarize_primary(original_runs)
original_tolerance = summarize_tolerances(original_runs)

pd.testing.assert_frame_equal(original_primary, released_primary, check_dtype=False)
pd.testing.assert_frame_equal(original_tolerance, released_tolerance, check_dtype=False)

print("PASS: released primary and tolerance summaries recompute exactly from A1_run_summary.csv.")
display(original_primary)


## Numerical-stability amendment

The original sweep computed leverage scores as squared row norms of the reduced QR factor $Q$. Twenty high-$p$ runs hit the 10,000-update cap. The final experiment instead evaluates the same exact leverage-score formula through $AR^{-1}$ and reruns all 750 cases.

The retained direct-$Q$ data below are diagnostic only; they are not used in the final A1 summaries.


In [ ]:
directq = pd.read_csv(DATA / "A1_run_summary_directQ_diagnostic.csv")
directq_failures = directq[directq["T_1e-8"].isna()]

print("Direct-Q capped cases:", len(directq_failures))
print(directq_failures.groupby("p").size().rename("capped"))


## Canonical Lewis update

For $r=1-2/p$, we form $B=W^{r/2}A=QR$. Since $Q=W^{r/2}AR^{-1}$, the leverage score of row $i$ can be evaluated as $w_i^r\|a_i^\top R^{-1}\|_2^2$. We compute $AR^{-1}$ by solving $R^\top X^\top=A^\top$ rather than explicitly forming $R^{-1}$.


In [ ]:
def lewis_update_rescaled_qr(A, w, p):
    r = 1.0 - 2.0 / p
    B = (w ** (r / 2.0))[:, None] * A
    _, R = np.linalg.qr(B, mode="reduced")
    X = np.linalg.solve(R.T, A.T).T  # X = A R^{-1}
    return (w ** r) * np.sum(X * X, axis=1)


def residual(w, w_new):
    return np.max(np.abs(np.log(w_new / w)))


def run_lewis(A, p, thresholds=THRESHOLDS, max_iter=MAX_ITER):
    m, n = A.shape
    w = np.full(m, n / m, dtype=np.float64)
    crossing = {tau: None for tau in thresholds}
    residuals = []

    for update_count in range(1, max_iter + 1):
        w_new = lewis_update_rescaled_qr(A, w, p)
        R = residual(w, w_new)
        residuals.append(R)

        for tau in thresholds:
            if crossing[tau] is None and R < tau:
                crossing[tau] = update_count

        w = w_new
        if R < min(thresholds):
            break

    summary = {
        "p": p,
        "T_1e-4": crossing[1e-4],
        "T_1e-6": crossing[1e-6],
        "T_1e-8": crossing[1e-8],
        "reached_primary_tol": crossing[PRIMARY_TOL] is not None,
        "updates_run": update_count,
        "final_residual": residuals[-1],
        "final_sum_w": w.sum(),
        "sum_w_error": abs(w.sum() - n),
    }
    return summary, np.asarray(residuals)


## Regenerate the matrix bank

We regenerate every matrix from its recorded seed and compare it with the released exact matrix bank. If a future NumPy version changes the random-normal implementation, we fall back to the released arrays for the numerical rerun so that the input matrices remain exactly those used in the original experiment.


In [ ]:
generated_matrices = np.stack([
    np.random.default_rng(seed).standard_normal((M, N))
    for seed in FINAL_SEEDS
]).astype(np.float64)

bank_exact = np.array_equal(generated_matrices, original_matrices)
print("Fresh seed generation is bit-identical to released matrix bank:", bank_exact)

if bank_exact:
    matrices_for_rerun = generated_matrices
else:
    print("WARNING: RNG output differs in this environment; using the released exact matrix bank for the rerun.")
    matrices_for_rerun = original_matrices.copy()

np.savez_compressed(
    REPRODUCED / "A1_matrix_bank.npz",
    seeds=np.array(FINAL_SEEDS, dtype=np.int64),
    matrices=matrices_for_rerun,
)

fresh_diag = pd.DataFrame({
    "seed": FINAL_SEEDS,
    "rank": [np.linalg.matrix_rank(A) for A in matrices_for_rerun],
    "cond_A": [np.linalg.cond(A) for A in matrices_for_rerun],
})
fresh_diag.to_csv(REPRODUCED / "A1_matrix_diagnostics.csv", index=False)

print("All ranks:", sorted(fresh_diag["rank"].unique()))
print("Condition-number range:", fresh_diag["cond_A"].min(), "to", fresh_diag["cond_A"].max())


## Fresh canonical rerun

This cell executes the complete $50\times15$ experiment with the rescaled-QR update. It writes fresh outputs only to `reproduced/`.


In [ ]:
if RUN_FRESH:
    cond_by_seed = dict(zip(fresh_diag["seed"], fresh_diag["cond_A"]))
    summary_rows = []
    residual_rows = []
    experiment_t0 = time.perf_counter()

    for matrix_id, (seed, A) in enumerate(zip(FINAL_SEEDS, matrices_for_rerun), start=1):
        for p in P_GRID:
            run_t0 = time.perf_counter()
            summary, rs = run_lewis(A, p)
            runtime_sec = time.perf_counter() - run_t0

            summary_rows.append({
                "matrix_id": matrix_id,
                "seed": seed,
                "p": p,
                "cond_A": cond_by_seed[seed],
                **summary,
                "runtime_sec": runtime_sec,
            })

            residual_rows.extend({
                "matrix_id": matrix_id,
                "seed": seed,
                "p": p,
                "update_count": k,
                "residual": R,
            } for k, R in enumerate(rs, start=1))

        if matrix_id % 10 == 0:
            print(f"completed {matrix_id}/50 matrices")

    fresh_runs = pd.DataFrame(summary_rows)
    fresh_residuals = pd.DataFrame(residual_rows)
    fresh_runs.to_csv(REPRODUCED / "A1_run_summary.csv", index=False)
    fresh_residuals.to_csv(REPRODUCED / "A1_residual_trajectories.csv", index=False)

    print("Fresh rerun complete in", round(time.perf_counter() - experiment_t0, 2), "seconds")
    print("Runs:", len(fresh_runs), "Residual rows:", len(fresh_residuals))
else:
    fresh_runs = None
    fresh_residuals = None
    print("Fresh rerun skipped; analysis below uses the released canonical data.")


## Compare the fresh rerun with the released canonical outputs

Stopping times can be sensitive to low-level floating-point differences at high $p$. We therefore report exact count agreement when it occurs, but we do not require byte-identical trajectories across numerical environments.


In [ ]:
if RUN_FRESH:
    assert len(fresh_runs) == 750
    assert fresh_runs[["matrix_id", "p"]].duplicated().sum() == 0
    assert fresh_runs["T_1e-8"].notna().all(), "At least one fresh run did not reach 1e-8 before the cap."

    key = ["matrix_id", "seed", "p"]
    compare = original_runs.merge(fresh_runs, on=key, suffixes=("_original", "_fresh"))

    for col in ["T_1e-4", "T_1e-6", "T_1e-8"]:
        delta = compare[f"{col}_fresh"] - compare[f"{col}_original"]
        print(
            col,
            "exact matches:", int((delta == 0).sum()), "/ 750;",
            "max |difference|:", float(delta.abs().max()),
        )

    fresh_primary = summarize_primary(fresh_runs)
    fresh_tolerance = summarize_tolerances(fresh_runs)
    fresh_primary.to_csv(REPRODUCED / "A1_primary_summary.csv", index=False)
    fresh_tolerance.to_csv(REPRODUCED / "A1_tolerance_summary.csv", index=False)

    median_delta = fresh_primary["median"] - original_primary["median"]
    print("Maximum absolute difference in primary medians:", float(median_delta.abs().max()))
else:
    fresh_primary = original_primary.copy()
    fresh_tolerance = original_tolerance.copy()


## Regenerate the final figure

The released paper figure uses the primary median with 20th/80th percentile curves on the left and the three median threshold curves on the right. We regenerate the figure from the fresh rerun when enabled; otherwise we use the released canonical summaries.


In [ ]:
def make_combined_figure(primary, tolerance, output_stem):
    BLUE = "#0072B2"
    MAGENTA = "#CC79A7"
    BLACK = "#222222"

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2))
    ax1, ax2 = axes

    ax1.plot(primary["p"], primary["median"], color=BLUE, marker="o",
             markersize=4.2, linewidth=2.0, label="Median")
    ax1.plot(primary["p"], primary["q20"], color=MAGENTA, linestyle="--",
             linewidth=1.5, alpha=0.8, label="20th percentile")
    ax1.plot(primary["p"], primary["q80"], color=BLACK, linestyle=":",
             linewidth=1.6, alpha=0.8, label="80th percentile")
    ax1.set_xlabel(r"$p$")
    ax1.set_ylabel(r"Iterations to reach $R_t < 10^{-8}$")
    ax1.grid(True, alpha=0.22)
    ax1.legend(frameon=True, fontsize=8.5)

    ax2.plot(tolerance["p"], tolerance["T_1e-4"], marker="o", markersize=4,
             linewidth=1.8, label=r"$10^{-4}$")
    ax2.plot(tolerance["p"], tolerance["T_1e-6"], marker="s", markersize=4,
             linewidth=1.8, label=r"$10^{-6}$")
    ax2.plot(tolerance["p"], tolerance["T_1e-8"], marker="^", markersize=4,
             linewidth=1.8, label=r"$10^{-8}$")
    ax2.set_xlabel(r"$p$")
    ax2.set_ylabel("Median iteration count")
    ax2.grid(True, alpha=0.22)
    ax2.legend(title="Threshold", frameon=True, fontsize=8.5, title_fontsize=8.5)

    fig.tight_layout()
    for ext in ["png", "svg", "pdf"]:
        kwargs = {"dpi": 300} if ext == "png" else {}
        fig.savefig(output_stem.with_suffix(f".{ext}"), bbox_inches="tight", **kwargs)
    return fig

plot_primary = fresh_primary if RUN_FRESH else original_primary
plot_tolerance = fresh_tolerance if RUN_FRESH else original_tolerance

fig = make_combined_figure(plot_primary, plot_tolerance, REPRODUCED / "A1_combined_panels")
plt.show()
print("Saved reproduced figure files in", REPRODUCED)


## Compact outcome check

The released canonical data contain 750 successful primary crossings. The primary median rises from 18 updates at $p=3$ to 286 at $p=32$, 581 at $p=64$, and 1163.5 at $p=127.3$. The same qualitative dependence on $p$ appears at the two looser thresholds.


In [ ]:
for p in [3, 32, 64, 127.3]:
    row = original_primary.loc[original_primary["p"] == p].iloc[0]
    print(f"p={p}: median T_1e-8 = {row['median']}")

print("\nPublic release files remain unchanged; fresh outputs are in reproduced/ only.")
